# LLM vote stability analysis

Notebook for evaluating One Night Ultimate Werewolf LLM voting runs.

Pipeline:

1. load project configuration and the cleaned game index;
2. build one ground-truth row per game;
3. load exactly three stochastic runs and one greedy run from `results/voting/<model_name>/`;
4. parse and evaluate all model votes;
5. compute run-level accuracy and status counts;
6. measure vote stability/agreement **only across the stochastic runs**;
7. compute majority-vote, at-least-one-correct, and greedy-vs-stochastic diagnostics;
8. sample useful examples by agreement/correctness type;
9. compute human, village-aligned, and LLM/human overlap analyses;
10. save reusable tables and printable examples under `analysis/<model_name>/<model_stage>/voting/`.

## Imports


In [1]:
from pathlib import Path
from typing import Any, Optional
from collections import Counter
from itertools import combinations
import json
import re

import numpy as np
import pandas as pd


## Configuration


In [2]:
# ---------------------------------------------------------------------
# Project and experiment configuration
# ---------------------------------------------------------------------
REPO_NAME = "masters_thesis_sdg"

# Input voting JSONs are expected under exactly:
#   results/voting/<LLM_NAME>/prompt_v4_run_1
#   results/voting/<LLM_NAME>/prompt_v4_run_2
#   results/voting/<LLM_NAME>/prompt_v4_run_3
#   results/voting/<LLM_NAME>/prompt_v4_t0        # greedy run
LLM_NAME = "unsloth_gemma-4-31B-it-unsloth-bnb-4bit"

# Used only for notebook-produced analysis outputs:
#   analysis/<LLM_NAME>/<MODEL_STAGE>/voting/...
# This does not affect the input voting-results path.
MODEL_STAGE = "base"

# Prompt family to evaluate. Accepts both "v4" and "prompt_v4".
PROMPT_FAMILY = "v4"

# The current analysis assumes exactly 3 stochastic runs plus 1 greedy run.
EXPECTED_STOCHASTIC_RUNS = (1, 2, 3)
EXPECT_GREEDY_RUN = True

# Conservative circle/no-werewolf labels. Add aliases here only if your parser
# really emits them as final votes.
CIRCLE_LABELS = {"no werewolf"}

# Roles excluded when computing the village-aligned vote preference.
NON_VILLAGE_ALIGNED_ROLES = {"Werewolf", "Minion", "Tanner"}

# Example sampling
RANDOM_STATE = 42
SAMPLE_PER_AGREEMENT_TYPE = 3

# Saving switches
SAVE_TABLES = True
SAVE_PRINTABLE_EXAMPLES = True

## Generic utilities and paths


In [3]:
def strip_prompt_prefix(prompt_family: str) -> str:
    """Accept both 'v4' and 'prompt_v4', returning 'v4'."""
    return str(prompt_family).removeprefix("prompt_")


def find_repo_root(start: Optional[Path] = None, repo_name: str = REPO_NAME) -> Path:
    """Walk upward until the repository root folder is found."""
    current = (start or Path.cwd()).resolve()

    while True:
        if current.name == repo_name:
            return current

        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root {repo_name!r} starting from {start or Path.cwd()}"
            )

        current = current.parent


def load_json(path: Path) -> Any:
    """Load a UTF-8 JSON file."""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: Path) -> None:
    """Save JSON with readable indentation."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


PROMPT_FAMILY_CLEAN = strip_prompt_prefix(PROMPT_FAMILY)
PROMPT_DIR_PREFIX = f"prompt_{PROMPT_FAMILY_CLEAN}"

REPO_ROOT = find_repo_root()
IDX_FILE_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "lai2023"
    / "onuw_transcripts_ready"
    / "index_cleaned.json"
)

# ---------------------------------------------------------------------
# Input voting results
# ---------------------------------------------------------------------
# Expected input layout:
#   results/voting/<model_name>/prompt_v4_run_1
#   results/voting/<model_name>/prompt_v4_run_2
#   results/voting/<model_name>/prompt_v4_run_3
#   results/voting/<model_name>/prompt_v4_t0

RESULTS_DIR = REPO_ROOT / "results" / "voting"
MODEL_RESULTS_ROOT = RESULTS_DIR / LLM_NAME
RUNS_ROOT = MODEL_RESULTS_ROOT

# ---------------------------------------------------------------------
# Notebook-produced analysis outputs
# ---------------------------------------------------------------------
# Expected output layout:
#   analysis/<model_name>/<stage>/voting/prompt_v4/vote_stability/...

ANALYSIS_DIR = REPO_ROOT / "analysis"
MODEL_ANALYSIS_ROOT = ANALYSIS_DIR / LLM_NAME
STAGE_ANALYSIS_ROOT = MODEL_ANALYSIS_ROOT / MODEL_STAGE
VOTING_ANALYSIS_ROOT = STAGE_ANALYSIS_ROOT / "voting"

OUT_DIR = VOTING_ANALYSIS_ROOT / PROMPT_DIR_PREFIX / "vote_stability"
TABLES_DIR = OUT_DIR / "tables"
EXAMPLES_DIR = OUT_DIR / "examples"

index_data = load_json(IDX_FILE_PATH)

print("REPO_ROOT:", REPO_ROOT)
print("IDX_FILE_PATH:", IDX_FILE_PATH)
print()
print("Input voting results")
print("RESULTS_DIR:", RESULTS_DIR)
print("MODEL_RESULTS_ROOT:", MODEL_RESULTS_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print()
print("Analysis outputs")
print("ANALYSIS_DIR:", ANALYSIS_DIR)
print("MODEL_ANALYSIS_ROOT:", MODEL_ANALYSIS_ROOT)
print("STAGE_ANALYSIS_ROOT:", STAGE_ANALYSIS_ROOT)
print("VOTING_ANALYSIS_ROOT:", VOTING_ANALYSIS_ROOT)
print("OUT_DIR:", OUT_DIR)
print()
print("Loaded games from index:", len(index_data))
print("LLM:", LLM_NAME)
print("Model stage:", MODEL_STAGE)
print("Prompt family:", PROMPT_FAMILY)

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
IDX_FILE_PATH: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed\lai2023\onuw_transcripts_ready\index_cleaned.json

Input voting results
RESULTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting
MODEL_RESULTS_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gemma-4-31B-it-unsloth-bnb-4bit
RUNS_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gemma-4-31B-it-unsloth-bnb-4bit

Analysis outputs
ANALYSIS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis
MODEL_ANALYSIS_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit
STAGE_ANALYSIS_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base
VOTING_ANALYSIS_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\ba

## Ground-truth table


In [19]:
def normalize_missing_vote(vote: Any) -> bool:
    """Return True when a raw human-vote value should be treated as missing."""
    if vote is None:
        return True

    if isinstance(vote, str):
        return vote.strip().upper() in {"", "NA", "N/A", "NONE", "NULL"}

    return False


def clean_human_votes(voting_outcome: Any) -> list[int]:
    """Convert a raw voting outcome into valid integer votes, ignoring missing values."""
    if voting_outcome is None:
        return []

    clean_votes = []

    for vote in voting_outcome:
        if normalize_missing_vote(vote):
            continue

        try:
            clean_votes.append(int(vote))
        except (TypeError, ValueError):
            continue

    return clean_votes


def summarize_human_vote(voting_outcome: Any) -> dict[str, Any]:
    """
    Summarize the raw human vote outcome.

    This summary is descriptive. LLM correctness is computed independently from
    final roles:
    - a player vote is correct if the selected player is a final Werewolf;
    - a circle vote is correct if there is no final Werewolf.
    """
    clean_votes = clean_human_votes(voting_outcome)
    n_missing_votes = 0 if voting_outcome is None else len(voting_outcome) - len(clean_votes)

    if not clean_votes:
        return {
            "human_vote_available": False,
            "human_vote_type": None,
            "human_target_indices": [],
            "human_vote_counts": {},
            "n_missing_human_votes": n_missing_votes,
        }

    vote_counts = Counter(clean_votes)
    max_votes = max(vote_counts.values())

    # ONUW rule: if nobody receives at least two votes, nobody is eliminated.
    if max_votes < 2:
        return {
            "human_vote_available": True,
            "human_vote_type": "circle",
            "human_target_indices": [],
            "human_vote_counts": dict(vote_counts),
            "n_missing_human_votes": n_missing_votes,
        }

    target_indices = [
        player_idx
        for player_idx, count in vote_counts.items()
        if count == max_votes
    ]

    return {
        "human_vote_available": True,
        "human_vote_type": "player",
        "human_target_indices": target_indices,
        "human_vote_counts": dict(vote_counts),
        "n_missing_human_votes": n_missing_votes,
    }


def build_ground_truth_df(index_data: list[dict[str, Any]]) -> pd.DataFrame:
    """Build one clean row per game from index_cleaned.json."""
    rows = []

    for game in index_data:
        source = game["source"]
        session_name = game["session_name"]
        game_key = game["game_key"]

        processed_txt_path = game.get("processed_txt_path")
        game_name = (
            Path(processed_txt_path).stem
            if processed_txt_path
            else f"{session_name}_{game_key}"
        )

        player_names = game["player_names"]
        start_roles = game["start_roles"]
        end_roles = game["end_roles"]
        voting_outcome = game.get("voting_outcome", [])

        correct_player_indices = [
            player_idx
            for player_idx, role in enumerate(end_roles)
            if role == "Werewolf"
        ]

        correct_player_names = [
            player_names[player_idx]
            for player_idx in correct_player_indices
        ]

        has_werewolf = len(correct_player_indices) > 0
        human_summary = summarize_human_vote(voting_outcome)

        human_target_names = [
            player_names[player_idx]
            for player_idx in human_summary["human_target_indices"]
            if 0 <= player_idx < len(player_names)
        ]

        game_id = f"{source} / {session_name} / {game_key}"

        rows.append({
            "game_id": game_id,
            "game_name": game_name,
            "source": source,
            "session_name": session_name,
            "game_key": game_key,
            "processed_txt_path": processed_txt_path,
            "num_players": game.get("num_players"),
            "num_turns": game.get("num_turns"),
            "num_unparsed_lines": game.get("num_unparsed_lines"),
            "num_dropped_lines": game.get("num_dropped_lines"),
            "player_names": player_names,
            "start_roles": start_roles,
            "end_roles": end_roles,
            "has_werewolf": has_werewolf,
            "n_werewolves": len(correct_player_indices),
            "correct_player_indices": correct_player_indices,
            "correct_player_names": correct_player_names,
            "correct_vote_type": "player" if has_werewolf else "circle",
            "voting_outcome": voting_outcome,
            "human_vote_available": human_summary["human_vote_available"],
            "human_vote_type": human_summary["human_vote_type"],
            "human_target_indices": human_summary["human_target_indices"],
            "human_target_names": human_target_names,
            "human_vote_counts": human_summary["human_vote_counts"],
            "n_missing_human_votes": human_summary["n_missing_human_votes"],
            "warning": game.get("warning", []),
        })

    return pd.DataFrame(rows)


gt_df = build_ground_truth_df(index_data)

print("Ground-truth games:", len(gt_df))
print("Unique game_name:", gt_df["game_name"].nunique())
print()

print("Correct vote type distribution, computed from END ROLES:")
display(gt_df["correct_vote_type"].value_counts(dropna=False).to_frame("n_games"))
print()

print("Number of Werewolves distribution, computed from END ROLES:")
display(gt_df["n_werewolves"].value_counts().sort_index().to_frame("n_games"))

display(gt_df.head())

Ground-truth games: 191
Unique game_name: 191

Correct vote type distribution, computed from END ROLES:


,n_games
correct_vote_type,
player,165
circle,26



Number of Werewolves distribution, computed from END ROLES:


,n_games
n_werewolves,
0,26
1,101
2,64


,game_id,game_name,source,session_name,game_key,processed_txt_path,num_players,num_turns,num_unparsed_lines,num_dropped_lines,...,correct_player_names,correct_vote_type,voting_outcome,human_vote_available,human_vote_type,human_target_indices,human_target_names,human_vote_counts,n_missing_human_votes,warning
0,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game1,data/processed/lai2023/onuw_transcripts_ready/...,5,112,0,0,...,[Justin],player,"[2, 2, 4, 4, 0]",True,player,"[2, 4]","[Paul, Mitchell]","{2: 2, 4: 2, 0: 1}",0,[]
1,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game2,data/processed/lai2023/onuw_transcripts_ready/...,5,157,0,0,...,[Laura],player,"[3, 3, 3, 4, 3]",True,player,[3],[James],"{3: 4, 4: 1}",0,[]
2,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game3,data/processed/lai2023/onuw_transcripts_ready/...,5,79,0,0,...,[Mitchell],player,"[1, 2, 3, 0, 0]",True,player,[0],[Justin],"{1: 1, 2: 1, 3: 1, 0: 2}",0,[]
3,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game4,data/processed/lai2023/onuw_transcripts_ready/...,5,148,0,0,...,[],circle,"[1, 2, 1, 1, 0]",True,player,[1],[Laura],"{1: 3, 2: 1, 0: 1}",0,[No 'Werewolf' role in end roles]
4,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#...,Youtube,Flashback##ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#9,Game5,data/processed/lai2023/onuw_transcripts_ready/...,5,85,0,0,...,"[Laura, Paul]",player,"[4, 4, 4, 4, 0]",True,player,[4],[Mitchell],"{4: 4, 0: 1}",0,[]


## Discover prompt-run directories


In [5]:
prompt_prefix = f"prompt_{strip_prompt_prefix(PROMPT_FAMILY)}"

prompt_run_dirs = [
    {
        "prompt_dir": RUNS_ROOT / f"{prompt_prefix}_run_{i}",
        "prompt_version": f"{prompt_prefix}_run_{i}",
        "decoding": "stochastic",
        "temperature": 1.0,
        "run_number": i,
        "run_label": f"run_{i}",
        "sort_key": i,
        "input_root": RUNS_ROOT,
    }
    for i in EXPECTED_STOCHASTIC_RUNS
]

if EXPECT_GREEDY_RUN:
    prompt_run_dirs.append({
        "prompt_dir": RUNS_ROOT / f"{prompt_prefix}_t0",
        "prompt_version": f"{prompt_prefix}_t0",
        "decoding": "greedy",
        "temperature": 0.0,
        "run_number": 0,
        "run_label": "greedy_t0",
        "sort_key": 999,
        "input_root": RUNS_ROOT,
    })

RUNS_ROOT_USED = RUNS_ROOT

run_dirs_df = pd.DataFrame([
    {
        "run_label": item["run_label"],
        "run_number": item["run_number"],
        "decoding": item["decoding"],
        "temperature": item["temperature"],
        "prompt_version": item["prompt_version"],
        "prompt_dir": str(item["prompt_dir"]),
        "exists": item["prompt_dir"].exists(),
        "input_root": str(item["input_root"]),
    }
    for item in prompt_run_dirs
])

print("RUNS_ROOT_USED:", RUNS_ROOT_USED)
display(run_dirs_df)

RUNS_ROOT_USED: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gemma-4-31B-it-unsloth-bnb-4bit


,run_label,run_number,decoding,temperature,prompt_version,prompt_dir,exists,input_root
0,run_1,1,stochastic,1.0,prompt_v4_run_1,C:\Users\annab\Documents\GitHub\masters_thesis...,True,C:\Users\annab\Documents\GitHub\masters_thesis...
1,run_2,2,stochastic,1.0,prompt_v4_run_2,C:\Users\annab\Documents\GitHub\masters_thesis...,True,C:\Users\annab\Documents\GitHub\masters_thesis...
2,run_3,3,stochastic,1.0,prompt_v4_run_3,C:\Users\annab\Documents\GitHub\masters_thesis...,True,C:\Users\annab\Documents\GitHub\masters_thesis...
3,greedy_t0,0,greedy,0.0,prompt_v4_t0,C:\Users\annab\Documents\GitHub\masters_thesis...,True,C:\Users\annab\Documents\GitHub\masters_thesis...


## Parse and evaluate LLM result files


In [6]:
def normalize_text(value: Any) -> Optional[str]:
    """Return a stripped string, or None for missing values."""
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def normalize_for_match(value: Any) -> Optional[str]:
    """Normalize text for simple case-insensitive comparisons."""
    text = normalize_text(value)
    return text.lower() if text is not None else None


def is_circle_vote_label(vote: Any) -> bool:
    """Detect a no-werewolf / circle vote."""
    vote_norm = normalize_for_match(vote)
    return vote_norm in CIRCLE_LABELS if vote_norm is not None else False


def match_player_name(vote: Any, player_names: list[str]) -> Optional[str]:
    """
    Match a vote to one of the game players.

    The exact name is preferred. A case-insensitive fallback is kept only to avoid
    losing valid parser outputs because of capitalization.
    """
    vote_text = normalize_text(vote)
    if vote_text is None:
        return None

    if vote_text in player_names:
        return vote_text

    vote_norm = vote_text.lower()
    for player in player_names:
        if str(player).strip().lower() == vote_norm:
            return player

    return None


# Ground-truth lookups. Keep the original fallback behavior because this notebook
# is meant to work with the existing result files.
GT_LOOKUPS = {
    "by_full_key": {},
    "by_session_game": {},
    "by_game_name": {},
}

for _, row in gt_df.iterrows():
    full_key = (row["source"], row["session_name"], row["game_key"])
    session_game_key = (row["session_name"], row["game_key"])

    GT_LOOKUPS["by_full_key"][full_key] = row
    GT_LOOKUPS["by_session_game"].setdefault(session_game_key, []).append(row)
    GT_LOOKUPS["by_game_name"][row["game_name"]] = row


def resolve_gt_row(res_data: dict[str, Any], res_file: Path) -> Optional[pd.Series]:
    """Find the ground-truth row for one result JSON."""
    source = res_data.get("source")
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")

    full_key = (source, session_name, game_key)
    if full_key in GT_LOOKUPS["by_full_key"]:
        return GT_LOOKUPS["by_full_key"][full_key]

    candidates = GT_LOOKUPS["by_session_game"].get((session_name, game_key), [])
    if len(candidates) == 1:
        return candidates[0]

    return GT_LOOKUPS["by_game_name"].get(Path(res_file).stem)


def extract_model_vote(res_data: dict[str, Any]) -> tuple[Any, bool, Any]:
    """
    Extract vote validity, chosen vote, and justification from one result JSON.

    Expected fields:
    - validation["chosen_vote"]
    - validation["is_valid"]
    - parsed_output["justification"]

    A few direct fallbacks are kept because older result files may use slightly
    different keys.
    """
    validation = res_data.get("validation", {}) or {}
    parsed_output = res_data.get("parsed_output", {}) or {}

    if not isinstance(validation, dict):
        validation = {}
    if not isinstance(parsed_output, dict):
        parsed_output = {}

    chosen_vote = (
        normalize_text(validation.get("chosen_vote"))
        or normalize_text(parsed_output.get("chosen_vote"))
        or normalize_text(res_data.get("chosen_vote"))
        or normalize_text(res_data.get("vote"))
        or normalize_text(res_data.get("final_vote"))
    )

    is_valid = validation.get("is_valid")
    if is_valid is None:
        is_valid = chosen_vote is not None

    justification = (
        parsed_output.get("justification")
        or parsed_output.get("reasoning")
        or parsed_output.get("explanation")
        or res_data.get("justification")
        or res_data.get("reasoning")
        or res_data.get("explanation")
    )

    return chosen_vote, bool(is_valid), justification


def evaluate_one_result_file(res_file: Path, run_info: dict[str, Any]) -> dict[str, Any]:
    """Evaluate one model result JSON against the final roles."""
    res_data = load_json(res_file)
    gt_row = resolve_gt_row(res_data, res_file)

    source = res_data.get("source")
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")
    chosen_vote_raw, is_valid_parse, justification = extract_model_vote(res_data)

    base_record = {
        "path": str(res_file),
        "run_label": run_info["run_label"],
        "run_number": run_info["run_number"],
        "decoding": run_info["decoding"],
        "temperature": run_info["temperature"],
        "prompt_version": run_info["prompt_version"],
        "source": source,
        "session_name": session_name,
        "game_key": game_key,
        "game_id": f"{source} / {session_name} / {game_key}",
        "chosen_vote_raw": chosen_vote_raw,
        "justification": justification,
    }

    if gt_row is None:
        return {
            **base_record,
            "game_name": Path(res_file).stem,
            "status": "missing_ground_truth",
            "chosen_vote": None,
            "canonical_player_vote": None,
            "chosen_player_name": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": np.nan,
            "is_circle_vote": False,
            "correct_vote_type": None,
            "has_werewolf": np.nan,
            "correct_player_names": None,
            "player_names": None,
            "end_roles": None,
        }

    player_names = gt_row["player_names"]
    end_roles = gt_row["end_roles"]
    has_werewolf = gt_row["has_werewolf"]

    base_record.update({
        "game_id": gt_row["game_id"],
        "game_name": gt_row["game_name"],
        "source": gt_row["source"],
        "session_name": gt_row["session_name"],
        "game_key": gt_row["game_key"],
        "player_names": player_names,
        "end_roles": end_roles,
        "has_werewolf": has_werewolf,
        "correct_vote_type": gt_row["correct_vote_type"],
        "correct_player_names": gt_row["correct_player_names"],
    })

    if not is_valid_parse or chosen_vote_raw is None:
        return {
            **base_record,
            "status": "failed_parse",
            "chosen_vote": None,
            "canonical_player_vote": None,
            "chosen_player_name": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": np.nan,
            "is_circle_vote": False,
        }

    chosen_vote = normalize_text(chosen_vote_raw)
    matched_player = match_player_name(chosen_vote, player_names)

    if matched_player is not None:
        chosen_player_index = player_names.index(matched_player)
        voted_player_end_role = end_roles[chosen_player_index]
        is_correct = voted_player_end_role == "Werewolf"

        return {
            **base_record,
            "status": "player_vote",
            "chosen_vote": chosen_vote,
            "canonical_player_vote": matched_player,
            "chosen_player_name": matched_player,
            "chosen_player_index": chosen_player_index,
            "voted_player_end_role": voted_player_end_role,
            "is_correct": bool(is_correct),
            "is_circle_vote": False,
        }

    if is_circle_vote_label(chosen_vote):
        return {
            **base_record,
            "status": "circle_vote",
            "chosen_vote": "No Werewolf",
            "canonical_player_vote": None,
            "chosen_player_name": None,
            "chosen_player_index": np.nan,
            "voted_player_end_role": None,
            "is_correct": bool(not has_werewolf),
            "is_circle_vote": True,
        }

    return {
        **base_record,
        "status": "invalid_vote",
        "chosen_vote": chosen_vote,
        "canonical_player_vote": None,
        "chosen_player_name": None,
        "chosen_player_index": np.nan,
        "voted_player_end_role": None,
        "is_correct": np.nan,
        "is_circle_vote": False,
    }


def load_all_vote_results(prompt_run_dirs: list[dict[str, Any]]) -> pd.DataFrame:
    """Load and evaluate all JSON files from all discovered runs."""
    rows = []

    for run_info in prompt_run_dirs:
        result_files = sorted(run_info["prompt_dir"].rglob("*.json"))

        print(
            f"{run_info['run_label']:>10} | "
            f"{run_info['decoding']:>10} | "
            f"{run_info['prompt_version']} | "
            f"{len(result_files)} files"
        )

        for res_file in result_files:
            rows.append(evaluate_one_result_file(res_file, run_info))

    return pd.DataFrame(rows)

## Run-level evaluation


In [7]:
def summarize_run(group: pd.DataFrame) -> pd.Series:
    """Compute accuracy and diagnostic counts for one run."""
    valid_mask = group["status"].isin(["player_vote", "circle_vote"])
    valid_group = group[valid_mask]

    correct_player_mask = (
        (valid_group["status"] == "player_vote")
        & (valid_group["is_correct"] == True)
    )
    correct_circle_mask = (
        (valid_group["status"] == "circle_vote")
        & (valid_group["is_correct"] == True)
    )

    total_correct = int(valid_group["is_correct"].sum())
    valid_votes_cast = int(valid_mask.sum())
    accuracy = total_correct / valid_votes_cast if valid_votes_cast > 0 else np.nan

    return pd.Series({
        "total_files_found": len(group),
        "total_games_evaluated": int((group["status"] != "missing_ground_truth").sum()),
        "missing_ground_truth": int((group["status"] == "missing_ground_truth").sum()),
        "correct_werewolf_votes": int(correct_player_mask.sum()),
        "correct_circle_votes": int(correct_circle_mask.sum()),
        "total_attempted_circles": int((valid_group["status"] == "circle_vote").sum()),
        "failed_parses": int((group["status"] == "failed_parse").sum()),
        "invalid_votes": int((group["status"] == "invalid_vote").sum()),
        "valid_votes_cast": valid_votes_cast,
        "total_llm_wins": total_correct,
        "accuracy": accuracy,
        "accuracy_percent": accuracy * 100 if pd.notna(accuracy) else np.nan,
        "circle_rate": (
            (valid_group["status"] == "circle_vote").mean()
            if len(valid_group) > 0
            else np.nan
        ),
        "non_circle_accuracy": (
            valid_group.loc[valid_group["status"] == "player_vote", "is_correct"].mean()
            if (valid_group["status"] == "player_vote").any()
            else np.nan
        ),
        "circle_accuracy": (
            valid_group.loc[valid_group["status"] == "circle_vote", "is_correct"].mean()
            if (valid_group["status"] == "circle_vote").any()
            else np.nan
        ),
    })


llm_vote_df = load_all_vote_results(prompt_run_dirs)

llm_run_summary_df = (
    llm_vote_df
    .groupby(
        ["prompt_version", "run_label", "run_number", "decoding", "temperature"],
        dropna=False,
    )
    .apply(summarize_run)
    .reset_index()
    .sort_values(["decoding", "run_number"])
)

status_counts_df = (
    llm_vote_df
    .groupby(["run_label", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(llm_run_summary_df)
print()
print("Status counts:")
display(status_counts_df)


     run_1 | stochastic | prompt_v4_run_1 | 191 files
     run_2 | stochastic | prompt_v4_run_2 | 191 files
     run_3 | stochastic | prompt_v4_run_3 | 191 files
 greedy_t0 |     greedy | prompt_v4_t0 | 191 files


C:\Users\annab\AppData\Local\Temp\ipykernel_11564\806191902.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_run)


,prompt_version,run_label,run_number,decoding,temperature,total_files_found,total_games_evaluated,missing_ground_truth,correct_werewolf_votes,correct_circle_votes,total_attempted_circles,failed_parses,invalid_votes,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,circle_rate,non_circle_accuracy,circle_accuracy
3,prompt_v4_t0,greedy_t0,0,greedy,0.0,191.0,191.0,0.0,75.0,9.0,24.0,0.0,0.0,191.0,84.0,0.439791,43.979058,0.125654,0.449102,0.375000
0,prompt_v4_run_1,run_1,1,stochastic,1.0,191.0,191.0,0.0,72.0,8.0,18.0,0.0,0.0,191.0,80.0,0.418848,41.884817,0.094241,0.416185,0.444444
1,prompt_v4_run_2,run_2,2,stochastic,1.0,191.0,191.0,0.0,71.0,11.0,20.0,0.0,0.0,191.0,82.0,0.429319,42.931937,0.104712,0.415205,0.550000
2,prompt_v4_run_3,run_3,3,stochastic,1.0,191.0,191.0,0.0,66.0,11.0,31.0,0.0,0.0,191.0,77.0,0.403141,40.314136,0.162304,0.412500,0.354839



Status counts:


status,run_label,circle_vote,player_vote
0,greedy_t0,24,167
1,run_1,18,173
2,run_2,20,171
3,run_3,31,160


## Agreement and stability across stochastic runs

In [8]:
def build_vote_matrix(
    llm_vote_df: pd.DataFrame,
    prompt_run_dirs: list[dict[str, Any]],
) -> tuple[pd.DataFrame, list[str], list[str]]:
    """Build a game-by-run matrix of valid model votes."""
    valid_vote_df = llm_vote_df[
        llm_vote_df["status"].isin(["player_vote", "circle_vote"])
        & llm_vote_df["game_id"].notna()
    ].copy()

    vote_matrix = valid_vote_df.pivot_table(
        index="game_id",
        columns="run_label",
        values="chosen_vote",
        aggfunc="first",
    )

    all_run_cols = [
        item["run_label"]
        for item in prompt_run_dirs
        if item["run_label"] in vote_matrix.columns
    ]

    stochastic_run_cols = [
        item["run_label"]
        for item in prompt_run_dirs
        if item["decoding"] == "stochastic"
        and item["run_label"] in vote_matrix.columns
    ]

    return vote_matrix.reindex(columns=all_run_cols), all_run_cols, stochastic_run_cols


def compute_pairwise_agreement(
    vote_matrix: pd.DataFrame,
    run_cols: list[str],
) -> pd.DataFrame:
    """Compute pairwise vote agreement between selected runs."""
    pairwise_rows = []

    for run_a, run_b in combinations(run_cols, 2):
        sub = vote_matrix[[run_a, run_b]].dropna()
        n_common_games = len(sub)
        n_agree = int((sub[run_a] == sub[run_b]).sum())

        pairwise_rows.append({
            "run_a": run_a,
            "run_b": run_b,
            "n_common_games": n_common_games,
            "n_agree": n_agree,
            "n_disagree": n_common_games - n_agree,
            "agreement_rate": n_agree / n_common_games if n_common_games else np.nan,
        })

    pairwise_df = pd.DataFrame(pairwise_rows)

    if pairwise_df.empty:
        return pd.DataFrame(columns=[
            "run_a",
            "run_b",
            "n_common_games",
            "n_agree",
            "n_disagree",
            "agreement_rate",
        ])

    return pairwise_df.sort_values("agreement_rate", ascending=False)


def agreement_type_from_n_unique(n_unique_votes: int) -> str:
    """Name the agreement pattern among the three stochastic runs."""
    if n_unique_votes == 1:
        return "unanimous"
    if n_unique_votes == 2:
        return "two_vs_one"
    if n_unique_votes == 3:
        return "all_different"
    return "other"


def build_complete_agreement_table(
    vote_matrix: pd.DataFrame,
    run_cols: list[str],
) -> pd.DataFrame:
    """Keep games with valid predictions in selected runs and label agreement patterns."""
    if not run_cols:
        return pd.DataFrame()

    complete = vote_matrix.dropna(subset=run_cols).copy()
    complete["n_unique_votes"] = complete[run_cols].nunique(axis=1)
    complete["all_same"] = complete["n_unique_votes"] == 1
    complete["agreement_type"] = complete["n_unique_votes"].map(agreement_type_from_n_unique)
    complete["vote_pattern"] = complete[run_cols].apply(
        lambda row: " | ".join(str(value) for value in row),
        axis=1,
    )

    return complete.reset_index()


vote_matrix, all_run_cols, stochastic_run_cols = build_vote_matrix(
    llm_vote_df,
    prompt_run_dirs,
)

# Agreement/disagreement is intentionally computed only across stochastic runs.
agreement_run_cols = stochastic_run_cols
run_cols = agreement_run_cols  # downstream agreement/example cells use stochastic runs by default

complete_agreement_df = build_complete_agreement_table(
    vote_matrix,
    agreement_run_cols,
)

pairwise_agreement_df = compute_pairwise_agreement(
    vote_matrix,
    agreement_run_cols,
)

agreement_type_summary_df = (
    complete_agreement_df["agreement_type"]
    .value_counts()
    .rename_axis("agreement_type")
    .reset_index(name="n_games")
    if not complete_agreement_df.empty
    else pd.DataFrame(columns=["agreement_type", "n_games"])
)

if not agreement_type_summary_df.empty:
    agreement_type_summary_df["percent"] = (
        agreement_type_summary_df["n_games"] / agreement_type_summary_df["n_games"].sum()
    )

disagreements_df = (
    complete_agreement_df[~complete_agreement_df["all_same"]]
    .sort_values(["n_unique_votes", "vote_pattern"], ascending=[False, True])
    .reset_index(drop=True)
    if not complete_agreement_df.empty
    else pd.DataFrame()
)

n_games = len(complete_agreement_df)
n_same = int(complete_agreement_df["all_same"].sum()) if n_games else 0
n_diff = n_games - n_same

print("Overall agreement among stochastic runs:")
print(f"All runs found: {all_run_cols}")
print(f"Runs used for agreement: {agreement_run_cols}")
print(f"Games with at least one valid prediction: {len(vote_matrix)}")
print(f"Games valid in all stochastic runs: {n_games}")

if n_games:
    print(f"Same vote in all stochastic runs: {n_same} / {n_games} ({n_same / n_games:.2%})")
    print(f"Different vote across stochastic runs: {n_diff} / {n_games} ({n_diff / n_games:.2%})")
else:
    print("No games were valid in all stochastic runs.")

print()
print("Agreement type counts:")
display(agreement_type_summary_df)

print()
print("Pairwise agreement among stochastic runs:")
display(pairwise_agreement_df)

print()
print(f"Stochastic disagreements: {len(disagreements_df)}")
display(disagreements_df.head(20))

Overall agreement among stochastic runs:
All runs found: ['run_1', 'run_2', 'run_3', 'greedy_t0']
Runs used for agreement: ['run_1', 'run_2', 'run_3']
Games with at least one valid prediction: 191
Games valid in all stochastic runs: 191
Same vote in all stochastic runs: 118 / 191 (61.78%)
Different vote across stochastic runs: 73 / 191 (38.22%)

Agreement type counts:


,agreement_type,n_games,percent
0,unanimous,118,0.617801
1,two_vs_one,60,0.314136
2,all_different,13,0.068063



Pairwise agreement among stochastic runs:


,run_a,run_b,n_common_games,n_agree,n_disagree,agreement_rate
0,run_1,run_2,191,144,47,0.753927
1,run_1,run_3,191,135,56,0.706806
2,run_2,run_3,191,135,56,0.706806



Stochastic disagreements: 73


run_label,game_id,run_1,run_2,run_3,greedy_t0,n_unique_votes,all_same,agreement_type,vote_pattern
0,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#9##Novem...,Alana,Caitlynn,Mike,Alana,3,False,all_different,Alana | Caitlynn | Mike
1,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#30##Febr...,Alysha,Mitchell,Justin,Justin,3,False,all_different,Alysha | Mitchell | Justin
2,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,James,Laura,No Werewolf,No Werewolf,3,False,all_different,James | Laura | No Werewolf
3,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#89##Nove...,Justin,James,No Werewolf,No Werewolf,3,False,all_different,Justin | James | No Werewolf
4,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#62##Marc...,Justin,James,Paul,Paul,3,False,all_different,Justin | James | Paul
5,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#4##July#...,Justin,Mitchell,No Werewolf,No Werewolf,3,False,all_different,Justin | Mitchell | No Werewolf
6,Youtube / One#Night#Ultimate#Love#Letter##ONE#...,Justin,Mitchell,No Werewolf,No Werewolf,3,False,all_different,Justin | Mitchell | No Werewolf
7,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#55##Janu...,Justin,No Werewolf,Caitlynn,Justin,3,False,all_different,Justin | No Werewolf | Caitlynn
8,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#64##Marc...,Justin,Paul,James,Paul,3,False,all_different,Justin | Paul | James
9,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#86##Nove...,Mitchell,Justin,Paul,Justin,3,False,all_different,Mitchell | Justin | Paul


## Agreement cases and sampled examples


In [9]:
def build_correctness_matrix(llm_vote_df: pd.DataFrame, run_cols: list[str]) -> pd.DataFrame:
    """Return a game-by-run table with True/False correctness values."""
    valid_vote_df = llm_vote_df[
        llm_vote_df["status"].isin(["player_vote", "circle_vote"])
        & llm_vote_df["game_id"].notna()
    ].copy()

    matrix = valid_vote_df.pivot_table(
        index="game_id",
        columns="run_label",
        values="is_correct",
        aggfunc="first",
    )

    return matrix.reindex(columns=run_cols)


def label_agreement_case(row: pd.Series, run_cols: list[str]) -> str:
    """Assign a compact agreement/correctness label to one game."""
    n_correct = sum(bool(row[f"correct_{run}"]) for run in run_cols)

    if row["all_same"] and n_correct == len(run_cols):
        return "same_vote_all_correct"
    if row["all_same"] and n_correct == 0:
        return "same_vote_all_wrong"
    if (not row["all_same"]) and n_correct == len(run_cols):
        return "different_votes_all_correct"
    if (not row["all_same"]) and n_correct == 0:
        return "different_votes_all_wrong"

    return "different_votes_mixed_correctness"


def build_agreement_cases(
    complete_agreement_df: pd.DataFrame,
    correctness_matrix: pd.DataFrame,
    run_cols: list[str],
) -> pd.DataFrame:
    """Combine stochastic vote-agreement patterns with correctness information."""
    if complete_agreement_df.empty:
        return complete_agreement_df.copy()

    correctness_wide = (
        correctness_matrix
        .loc[complete_agreement_df["game_id"]]
        .add_prefix("correct_")
        .reset_index()
    )

    cases = complete_agreement_df.merge(correctness_wide, on="game_id", how="left")
    correct_cols = [f"correct_{run}" for run in run_cols]
    cases["n_correct_runs"] = cases[correct_cols].sum(axis=1).astype(int)
    cases["agreement_case"] = cases.apply(
        lambda row: label_agreement_case(row, run_cols),
        axis=1,
    )

    return cases


def sample_examples_by_agreement_case(
    agreement_cases_df: pd.DataFrame,
    n: int = SAMPLE_PER_AGREEMENT_TYPE,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Sample up to n examples for each agreement/correctness case."""
    if agreement_cases_df.empty:
        return agreement_cases_df.copy()

    return (
        agreement_cases_df
        .groupby("agreement_case", group_keys=False)
        .apply(lambda group: group.sample(min(n, len(group)), random_state=random_state))
        .sort_values(["agreement_case", "game_id"])
        .reset_index(drop=True)
    )


def get_game_votes(game_id: str, llm_vote_df: pd.DataFrame) -> pd.DataFrame:
    """Return all run-level rows for one game, ordered by discovered run order."""
    cols = [
        "run_label",
        "decoding",
        "prompt_version",
        "status",
        "chosen_vote",
        "canonical_player_vote",
        "chosen_player_name",
        "voted_player_end_role",
        "is_correct",
        "justification",
        "path",
    ]

    available_cols = [col for col in cols if col in llm_vote_df.columns]

    vote_rows = llm_vote_df.loc[
        llm_vote_df["game_id"] == game_id,
        available_cols,
    ].copy()

    run_order = {run: idx for idx, run in enumerate(all_run_cols)}
    vote_rows["_run_order"] = vote_rows["run_label"].map(run_order).fillna(999)

    return (
        vote_rows
        .sort_values(["_run_order", "run_label"])
        .drop(columns=["_run_order"])
        .reset_index(drop=True)
    )


def format_game_example(game_id: str, include_justifications: bool = True) -> str:
    """Format one game example as clean, readable plain text."""
    gt_rows = gt_df.loc[gt_df["game_id"] == game_id]
    if gt_rows.empty:
        return f"GAME: {game_id}\nGround truth not found."

    gt_row = gt_rows.iloc[0]
    vote_rows = get_game_votes(game_id, llm_vote_df)

    player_names = gt_row["player_names"]
    start_roles = gt_row["start_roles"]
    end_roles = gt_row["end_roles"]

    lines = [
        "=" * 88,
        f"GAME: {game_id}",
        "-" * 88,
        f"Correct vote type: {gt_row['correct_vote_type']}",
        f"Correct player(s): {gt_row['correct_player_names']}",
        "",
        "PLAYERS",
    ]

    for idx, player in enumerate(player_names):
        start_role = start_roles[idx] if idx < len(start_roles) else "?"
        end_role = end_roles[idx] if idx < len(end_roles) else "?"
        marker = "  <-- correct target" if player in gt_row["correct_player_names"] else ""
        lines.append(f"  {idx}. {player}: {start_role} -> {end_role}{marker}")

    lines.extend(["", "MODEL VOTES"])

    for _, vote_row in vote_rows.iterrows():
        matched_player = vote_row.get("canonical_player_vote")
        if pd.isna(matched_player):
            matched_player = None

        matched_text = ""
        if matched_player is not None:
            matched_text = f" | matched={matched_player} ({vote_row.get('voted_player_end_role')})"

        lines.append(
            f"  {vote_row['run_label']:<10} "
            f"[{vote_row['decoding']:<10}] "
            f"vote={vote_row['chosen_vote']} | "
            f"status={vote_row['status']} | "
            f"correct={vote_row['is_correct']}"
            f"{matched_text}"
        )

    if include_justifications:
        lines.extend(["", "JUSTIFICATIONS"])

        for _, vote_row in vote_rows.iterrows():
            justification = vote_row.get("justification")
            if justification is None or pd.isna(justification):
                justification = "<missing>"

            lines.extend([
                "",
                f"{vote_row['run_label']} [{vote_row['decoding']}]",
                str(justification).strip(),
            ])

    return "\n".join(lines)


def print_game_example(game_id: str, include_justifications: bool = True) -> None:
    """Print a detailed example for manual inspection."""
    print(format_game_example(game_id, include_justifications=include_justifications))


correctness_matrix = build_correctness_matrix(llm_vote_df, run_cols)
agreement_cases_df = build_agreement_cases(complete_agreement_df, correctness_matrix, run_cols)
sampled_agreement_examples_df = sample_examples_by_agreement_case(agreement_cases_df)

print("Agreement/correctness case counts:")
if "agreement_case" in agreement_cases_df.columns:
    display(agreement_cases_df["agreement_case"].value_counts().to_frame("n_games"))
else:
    print("No agreement cases available.")

print()
print("Sampled examples by agreement/correctness case:")
display(sampled_agreement_examples_df)

# Example manual inspection:
# game_id = sampled_agreement_examples_df.iloc[0]["game_id"]
# print_game_example(game_id)

Agreement/correctness case counts:


C:\Users\annab\AppData\Local\Temp\ipykernel_11564\1438692018.py:77: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(min(n, len(group)), random_state=random_state))


,n_games
agreement_case,
same_vote_all_wrong,64
same_vote_all_correct,54
different_votes_mixed_correctness,49
different_votes_all_wrong,22
different_votes_all_correct,2



Sampled examples by agreement/correctness case:


run_label,game_id,run_1,run_2,run_3,greedy_t0,n_unique_votes,all_same,agreement_type,vote_pattern,correct_run_1,correct_run_2,correct_run_3,n_correct_runs,agreement_case
0,Ego4D / 62c4bc58-3776-4791-ac30-4c9ca5619503 /...,Jessica,Jessica,Daniel,Jessica,2,False,two_vs_one,Jessica | Jessica | Daniel,True,True,True,3,different_votes_all_correct
1,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#7##Novem...,Mitchell,Caitlynn,Caitlynn,Caitlynn,2,False,two_vs_one,Mitchell | Caitlynn | Caitlynn,True,True,True,3,different_votes_all_correct
2,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,Sian,James,James,Sian,2,False,two_vs_one,Sian | James | James,False,False,False,0,different_votes_all_wrong
3,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,James,Laura,No Werewolf,No Werewolf,3,False,all_different,James | Laura | No Werewolf,False,False,False,0,different_votes_all_wrong
4,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF#13##July...,No Werewolf,Julie,No Werewolf,Julie,2,False,two_vs_one,No Werewolf | Julie | No Werewolf,False,False,False,0,different_votes_all_wrong
5,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#5...,Mitchell,No Werewolf,Mitchell,No Werewolf,2,False,two_vs_one,Mitchell | No Werewolf | Mitchell,False,True,False,1,different_votes_mixed_correctness
6,Youtube / Retrovision##ONE#NIGHT#ULTIMATE#WERE...,Mike,Mike,Eric,Mike,2,False,two_vs_one,Mike | Mike | Eric,True,True,False,2,different_votes_mixed_correctness
7,Youtube / Way#Way#Back##ONE#NIGHT#ULTIMATE#WER...,Laura,Laura,No Werewolf,James,2,False,two_vs_one,Laura | Laura | No Werewolf,True,True,False,2,different_votes_mixed_correctness
8,Youtube / ONE#NIGHT#ULTIMATE#WEREWOLF##Retro#3...,No Werewolf,No Werewolf,No Werewolf,No Werewolf,1,True,unanimous,No Werewolf | No Werewolf | No Werewolf,True,True,True,3,same_vote_all_correct
9,Youtube / Visions#from#2016##ONE#NIGHT#ULTIMAT...,Paul,Paul,Paul,Paul,1,True,unanimous,Paul | Paul | Paul,True,True,True,3,same_vote_all_correct


## Majority vote, at-least-one-correct, and greedy-vs-stochastic diagnostics

In [10]:

def vote_is_correct_for_game(game_id: str, vote: Any) -> Any:
    """Evaluate an arbitrary vote string against the final roles for one game."""
    if pd.isna(vote):
        return np.nan

    gt_rows = gt_df.loc[gt_df["game_id"] == game_id]
    if gt_rows.empty:
        return np.nan

    gt_row = gt_rows.iloc[0]

    if is_circle_vote_label(vote):
        return bool(not gt_row["has_werewolf"])

    matched_player = match_player_name(vote, gt_row["player_names"])
    if matched_player is None:
        return np.nan

    player_idx = gt_row["player_names"].index(matched_player)
    return bool(gt_row["end_roles"][player_idx] == "Werewolf")


def majority_vote_from_row(row: pd.Series, run_cols: list[str]) -> tuple[Any, bool, int]:
    """Return majority vote, whether a strict majority exists, and n unique votes."""
    votes = [row[col] for col in run_cols if pd.notna(row[col])]
    if not votes:
        return np.nan, False, 0

    counts = Counter(votes)
    max_count = max(counts.values())
    top_votes = [vote for vote, count in counts.items() if count == max_count]
    has_majority = len(top_votes) == 1 and max_count > len(votes) / 2

    return (top_votes[0] if has_majority else np.nan), has_majority, len(counts)


# Majority vote among the three stochastic runs.
stochastic_majority_df = complete_agreement_df[
    ["game_id", *agreement_run_cols, "agreement_type", "n_unique_votes", "vote_pattern"]
].copy()

majority_info = stochastic_majority_df.apply(
    lambda row: majority_vote_from_row(row, agreement_run_cols),
    axis=1,
    result_type="expand",
)
majority_info.columns = ["stochastic_majority_vote", "has_stochastic_majority", "n_unique_stochastic_votes"]
stochastic_majority_df = pd.concat([stochastic_majority_df, majority_info], axis=1)
stochastic_majority_df["stochastic_majority_correct"] = stochastic_majority_df.apply(
    lambda row: vote_is_correct_for_game(row["game_id"], row["stochastic_majority_vote"])
    if row["has_stochastic_majority"]
    else np.nan,
    axis=1,
)

# At least one stochastic run correct.
stochastic_correctness_df = correctness_matrix.dropna(subset=agreement_run_cols).copy()
stochastic_correctness_df["at_least_one_stochastic_correct"] = stochastic_correctness_df[agreement_run_cols].astype(bool).any(axis=1)
stochastic_correctness_df["all_stochastic_correct"] = stochastic_correctness_df[agreement_run_cols].astype(bool).all(axis=1)
stochastic_correctness_df = stochastic_correctness_df.reset_index()

# Greedy vs stochastic set / majority.
greedy_col = "greedy_t0" if "greedy_t0" in vote_matrix.columns else None

if greedy_col is not None:
    greedy_vs_stochastic_df = (
        vote_matrix
        .dropna(subset=[*agreement_run_cols, greedy_col])
        .reset_index()[["game_id", *agreement_run_cols, greedy_col]]
        .merge(
            stochastic_majority_df[[
                "game_id",
                "stochastic_majority_vote",
                "has_stochastic_majority",
                "stochastic_majority_correct",
                "agreement_type",
            ]],
            on="game_id",
            how="left",
        )
    )

    greedy_vs_stochastic_df["greedy_vote"] = greedy_vs_stochastic_df[greedy_col]
    greedy_vs_stochastic_df["greedy_in_stochastic_vote_set"] = greedy_vs_stochastic_df.apply(
        lambda row: row["greedy_vote"] in {row[col] for col in agreement_run_cols},
        axis=1,
    )
    greedy_vs_stochastic_df["greedy_matches_stochastic_majority"] = (
        greedy_vs_stochastic_df["has_stochastic_majority"]
        & (greedy_vs_stochastic_df["greedy_vote"] == greedy_vs_stochastic_df["stochastic_majority_vote"])
    )
    greedy_vs_stochastic_df["greedy_correct"] = greedy_vs_stochastic_df.apply(
        lambda row: vote_is_correct_for_game(row["game_id"], row["greedy_vote"]),
        axis=1,
    )
else:
    greedy_vs_stochastic_df = pd.DataFrame()

stochastic_summary = llm_run_summary_df[llm_run_summary_df["decoding"] == "stochastic"]
greedy_summary = llm_run_summary_df[llm_run_summary_df["run_label"] == "greedy_t0"]

aggregation_summary_rows = [
    {
        "metric": "single_stochastic_run_mean_accuracy",
        "n_games": int(stochastic_summary["valid_votes_cast"].mean()) if not stochastic_summary.empty else np.nan,
        "accuracy": stochastic_summary["accuracy"].mean() if not stochastic_summary.empty else np.nan,
    },
    {
        "metric": "stochastic_majority_accuracy_available_only",
        "n_games": int(stochastic_majority_df["has_stochastic_majority"].sum()),
        "accuracy": stochastic_majority_df.loc[
            stochastic_majority_df["has_stochastic_majority"],
            "stochastic_majority_correct",
        ].mean(),
    },
    {
        "metric": "stochastic_majority_accuracy_missing_majority_as_wrong",
        "n_games": len(stochastic_majority_df),
        "accuracy": stochastic_majority_df["stochastic_majority_correct"].fillna(False).mean(),
    },
    {
        "metric": "greedy_accuracy",
        "n_games": int(greedy_summary["valid_votes_cast"].iloc[0]) if not greedy_summary.empty else np.nan,
        "accuracy": greedy_summary["accuracy"].iloc[0] if not greedy_summary.empty else np.nan,
    },
    {
        "metric": "at_least_one_stochastic_run_correct",
        "n_games": len(stochastic_correctness_df),
        "accuracy": stochastic_correctness_df["at_least_one_stochastic_correct"].mean(),
    },
]

aggregation_summary_df = pd.DataFrame(aggregation_summary_rows)
aggregation_summary_df["accuracy_percent"] = aggregation_summary_df["accuracy"] * 100

if not greedy_vs_stochastic_df.empty:
    greedy_vs_stochastic_summary_df = pd.DataFrame([
        {
            "metric": "greedy_in_stochastic_vote_set",
            "n_games": len(greedy_vs_stochastic_df),
            "rate": greedy_vs_stochastic_df["greedy_in_stochastic_vote_set"].mean(),
        },
        {
            "metric": "greedy_matches_stochastic_majority_when_majority_exists",
            "n_games": int(greedy_vs_stochastic_df["has_stochastic_majority"].sum()),
            "rate": greedy_vs_stochastic_df.loc[
                greedy_vs_stochastic_df["has_stochastic_majority"],
                "greedy_matches_stochastic_majority",
            ].mean(),
        },
    ])
    greedy_vs_stochastic_summary_df["rate_percent"] = greedy_vs_stochastic_summary_df["rate"] * 100
else:
    greedy_vs_stochastic_summary_df = pd.DataFrame(columns=["metric", "n_games", "rate", "rate_percent"])

print("Aggregation summary:")
display(aggregation_summary_df)

print()
print("Greedy vs stochastic summary:")
display(greedy_vs_stochastic_summary_df)

print()
print("Stochastic majority examples:")
display(stochastic_majority_df.head())


Aggregation summary:


C:\Users\annab\AppData\Local\Temp\ipykernel_11564\1777371143.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "accuracy": stochastic_majority_df["stochastic_majority_correct"].fillna(False).mean(),


,metric,n_games,accuracy,accuracy_percent
0,single_stochastic_run_mean_accuracy,191,0.417103,41.710297
1,stochastic_majority_accuracy_available_only,178,0.432584,43.258427
2,stochastic_majority_accuracy_missing_majority_...,191,0.403141,40.314136
3,greedy_accuracy,191,0.439791,43.979058
4,at_least_one_stochastic_run_correct,191,0.549738,54.973822



Greedy vs stochastic summary:


,metric,n_games,rate,rate_percent
0,greedy_in_stochastic_vote_set,191,0.890052,89.005236
1,greedy_matches_stochastic_majority_when_majori...,178,0.758427,75.842697



Stochastic majority examples:


,game_id,run_1,run_2,run_3,agreement_type,n_unique_votes,vote_pattern,stochastic_majority_vote,has_stochastic_majority,n_unique_stochastic_votes,stochastic_majority_correct
0,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,Erin,Brent,Erin,two_vs_one,2,Erin | Brent | Erin,Erin,True,2,False
1,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,Jack,Jack,Jack,unanimous,1,Jack | Jack | Jack,Jack,True,1,True
2,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,Elliot,Elliot,Elliot,unanimous,1,Elliot | Elliot | Elliot,Elliot,True,1,False
3,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,Sian,Sian,Sian,unanimous,1,Sian | Sian | Sian,Sian,True,1,False
4,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,Elliot,Elliot,Elliot,unanimous,1,Elliot | Elliot | Elliot,Elliot,True,1,True


## Human baseline


In [12]:
def is_incomplete_human_vote(raw_votes: Any) -> bool:
    """
    Keep the stricter human-baseline behavior from the original notebook:
    a game is skipped when any raw human vote is missing.
    """
    if raw_votes is None:
        return True

    if isinstance(raw_votes, str):
        return raw_votes.strip().upper() in {"NA", "N/A", ""}

    if isinstance(raw_votes, list):
        if not raw_votes:
            return True

        normalized = [str(vote).strip().upper() for vote in raw_votes]
        return any(vote in {"NA", "N/A", ""} for vote in normalized)

    return False


def compute_human_baseline(index_data: list[dict[str, Any]]) -> pd.DataFrame:
    """Compute the descriptive human win rate using the original notebook logic."""
    valid_human_games = 0
    correct_werewolf_kills = 0
    correct_circle_votes = 0
    attempted_circle_votes = 0

    for game in index_data:
        raw_votes = game.get("voting_outcome", [])

        if is_incomplete_human_vote(raw_votes):
            continue

        votes = [int(vote) for vote in raw_votes]
        end_roles = game.get("end_roles", [])

        valid_human_games += 1
        vote_counts = Counter(votes)

        if not vote_counts:
            continue

        max_votes = max(vote_counts.values())

        if max_votes >= 2:
            eliminated_players = [
                player_idx
                for player_idx, count in vote_counts.items()
                if count == max_votes
            ]

            wolf_killed = any(
                end_roles[player_idx] == "Werewolf"
                for player_idx in eliminated_players
            )

            if wolf_killed:
                correct_werewolf_kills += 1

        else:
            attempted_circle_votes += 1

            if "Werewolf" not in end_roles:
                correct_circle_votes += 1

    total_human_wins = correct_werewolf_kills + correct_circle_votes
    human_win_rate = (
        total_human_wins / valid_human_games
        if valid_human_games > 0
        else np.nan
    )

    return pd.DataFrame([{
        "total_games_in_index": len(index_data),
        "valid_human_games": valid_human_games,
        "correct_werewolf_kills": correct_werewolf_kills,
        "correct_circle_votes": correct_circle_votes,
        "total_attempted_circles": attempted_circle_votes,
        "total_human_wins": total_human_wins,
        "human_win_rate": human_win_rate,
        "human_win_rate_percent": human_win_rate * 100,
    }])


human_summary_df = compute_human_baseline(index_data)
human_win_rate = human_summary_df.loc[0, "human_win_rate"]

display(human_summary_df)


,total_games_in_index,valid_human_games,correct_werewolf_kills,correct_circle_votes,total_attempted_circles,total_human_wins,human_win_rate,human_win_rate_percent
0,191,171,77,7,12,84,0.491228,49.122807


## LLM vs human comparison


In [13]:
comparison_with_human_df = llm_run_summary_df[
    [
        "prompt_version",
        "run_label",
        "decoding",
        "temperature",
        "valid_votes_cast",
        "total_llm_wins",
        "accuracy",
        "accuracy_percent",
        "failed_parses",
        "invalid_votes",
        "total_attempted_circles",
    ]
].copy()

comparison_with_human_df["human_win_rate"] = human_win_rate
comparison_with_human_df["human_win_rate_percent"] = human_win_rate * 100
comparison_with_human_df["delta_vs_human_percent"] = (
    comparison_with_human_df["accuracy_percent"]
    - comparison_with_human_df["human_win_rate_percent"]
)

display(comparison_with_human_df)


,prompt_version,run_label,decoding,temperature,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,failed_parses,invalid_votes,total_attempted_circles,human_win_rate,human_win_rate_percent,delta_vs_human_percent
3,prompt_v4_t0,greedy_t0,greedy,0.0,191.0,84.0,0.439791,43.979058,0.0,0.0,24.0,0.491228,49.122807,-5.143749
0,prompt_v4_run_1,run_1,stochastic,1.0,191.0,80.0,0.418848,41.884817,0.0,0.0,18.0,0.491228,49.122807,-7.237990
1,prompt_v4_run_2,run_2,stochastic,1.0,191.0,82.0,0.429319,42.931937,0.0,0.0,20.0,0.491228,49.122807,-6.190870
2,prompt_v4_run_3,run_3,stochastic,1.0,191.0,77.0,0.403141,40.314136,0.0,0.0,31.0,0.491228,49.122807,-8.808671


## Village-aligned vote dispersion and LLM/human overlap

In [14]:

def shannon_entropy_from_counts(counts: Counter) -> float:
    """Compute Shannon entropy from a Counter."""
    total = sum(counts.values())
    if total == 0:
        return np.nan

    probs = np.array([count / total for count in counts.values()], dtype=float)
    return float(-(probs * np.log(probs)).sum())


def parse_indexed_votes(raw_votes: Any) -> list[Optional[int]]:
    """Parse vote indices while preserving voter position."""
    if not isinstance(raw_votes, list):
        return []

    parsed = []
    for vote in raw_votes:
        if normalize_missing_vote(vote):
            parsed.append(None)
            continue

        try:
            parsed.append(int(vote))
        except (TypeError, ValueError):
            parsed.append(None)

    return parsed


def summarize_actual_human_outcome(row: pd.Series) -> dict[str, Any]:
    """Compute the mechanical ONUW outcome using all human votes."""
    raw_votes = row["voting_outcome"]

    if is_incomplete_human_vote(raw_votes):
        return {
            "game_id": row["game_id"],
            "actual_human_available": False,
            "actual_human_vote_type": None,
            "actual_human_targets": [],
            "actual_human_target_names": [],
            "actual_human_win": np.nan,
        }

    votes = [vote for vote in parse_indexed_votes(raw_votes) if vote is not None]
    vote_counts = Counter(votes)

    if not vote_counts:
        return {
            "game_id": row["game_id"],
            "actual_human_available": False,
            "actual_human_vote_type": None,
            "actual_human_targets": [],
            "actual_human_target_names": [],
            "actual_human_win": np.nan,
        }

    max_votes = max(vote_counts.values())

    if max_votes < 2:
        return {
            "game_id": row["game_id"],
            "actual_human_available": True,
            "actual_human_vote_type": "circle",
            "actual_human_targets": [],
            "actual_human_target_names": [],
            "actual_human_win": bool(not row["has_werewolf"]),
        }

    targets = [idx for idx, count in vote_counts.items() if count == max_votes]
    valid_targets = [idx for idx in targets if 0 <= idx < len(row["player_names"])]
    target_names = [row["player_names"][idx] for idx in valid_targets]
    human_win = any(row["end_roles"][idx] == "Werewolf" for idx in valid_targets)

    return {
        "game_id": row["game_id"],
        "actual_human_available": True,
        "actual_human_vote_type": "player",
        "actual_human_targets": valid_targets,
        "actual_human_target_names": target_names,
        "actual_human_win": bool(human_win),
    }


def summarize_village_aligned_votes(row: pd.Series) -> dict[str, Any]:
    """
    Summarize the votes cast by players whose final role is aligned with Village.

    Excludes Werewolf, Minion, and Tanner voters. This is not the mechanical game
    outcome; it is a descriptive proxy for village-aligned voting preference.
    """
    parsed_votes = parse_indexed_votes(row["voting_outcome"])
    player_names = row["player_names"]
    end_roles = row["end_roles"]

    village_votes = []
    village_voters = []

    for voter_idx, target_idx in enumerate(parsed_votes):
        if voter_idx >= len(end_roles):
            continue
        if end_roles[voter_idx] in NON_VILLAGE_ALIGNED_ROLES:
            continue
        if target_idx is None or not (0 <= target_idx < len(player_names)):
            continue

        village_voters.append(player_names[voter_idx])
        village_votes.append(target_idx)

    vote_counts = Counter(village_votes)
    n_votes = sum(vote_counts.values())

    if n_votes == 0:
        return {
            "game_id": row["game_id"],
            "n_village_aligned_votes": 0,
            "village_vote_counts": {},
            "village_top_targets": [],
            "village_top_target_names": [],
            "village_top_share": np.nan,
            "village_vote_entropy": np.nan,
            "village_modal_correct": np.nan,
        }

    max_votes = max(vote_counts.values())
    top_targets = [idx for idx, count in vote_counts.items() if count == max_votes]
    top_target_names = [player_names[idx] for idx in top_targets]

    # A tied modal preference is counted as correct if at least one modal target is a Werewolf.
    village_modal_correct = any(end_roles[idx] == "Werewolf" for idx in top_targets)

    return {
        "game_id": row["game_id"],
        "n_village_aligned_votes": n_votes,
        "village_vote_counts": {player_names[idx]: count for idx, count in vote_counts.items()},
        "village_top_targets": top_targets,
        "village_top_target_names": top_target_names,
        "village_top_share": max_votes / n_votes,
        "village_vote_entropy": shannon_entropy_from_counts(vote_counts),
        "village_modal_correct": bool(village_modal_correct),
    }


def row_vote_entropy(row: pd.Series, run_cols: list[str]) -> float:
    """Entropy of the stochastic LLM votes for one game."""
    votes = [row[col] for col in run_cols if pd.notna(row[col])]
    return shannon_entropy_from_counts(Counter(votes))


def row_top_share(row: pd.Series, run_cols: list[str]) -> float:
    """Top-vote share among the stochastic LLM votes for one game."""
    votes = [row[col] for col in run_cols if pd.notna(row[col])]
    counts = Counter(votes)
    total = sum(counts.values())
    return max(counts.values()) / total if total else np.nan


human_per_game_df = gt_df.apply(
    lambda row: pd.Series(summarize_actual_human_outcome(row)),
    axis=1,
)

village_vote_df = gt_df.apply(
    lambda row: pd.Series(summarize_village_aligned_votes(row)),
    axis=1,
)

llm_stability_df = complete_agreement_df[["game_id", "agreement_type", *agreement_run_cols]].copy()
llm_stability_df["llm_vote_entropy"] = llm_stability_df.apply(
    lambda row: row_vote_entropy(row, agreement_run_cols),
    axis=1,
)
llm_stability_df["llm_top_share"] = llm_stability_df.apply(
    lambda row: row_top_share(row, agreement_run_cols),
    axis=1,
)

llm_per_game_df = stochastic_majority_df[[
    "game_id",
    "agreement_type",
    "stochastic_majority_vote",
    "has_stochastic_majority",
    "stochastic_majority_correct",
]].copy()

if not greedy_vs_stochastic_df.empty:
    llm_per_game_df = llm_per_game_df.merge(
        greedy_vs_stochastic_df[["game_id", "greedy_vote", "greedy_correct"]],
        on="game_id",
        how="left",
    )
else:
    llm_per_game_df["greedy_vote"] = np.nan
    llm_per_game_df["greedy_correct"] = np.nan

human_llm_game_df = (
    human_per_game_df
    .merge(village_vote_df, on="game_id", how="left")
    .merge(llm_stability_df[["game_id", "llm_vote_entropy", "llm_top_share"]], on="game_id", how="left")
    .merge(llm_per_game_df, on="game_id", how="left")
)

corr_df = human_llm_game_df.dropna(subset=["llm_vote_entropy", "village_vote_entropy", "village_top_share"])

village_llm_correlation_df = pd.DataFrame([
    {
        "x": "village_vote_entropy",
        "y": "llm_vote_entropy",
        "n_games": len(corr_df),
        "pearson_corr": corr_df["village_vote_entropy"].corr(corr_df["llm_vote_entropy"]),
    },
    {
        "x": "village_top_share",
        "y": "llm_vote_entropy",
        "n_games": len(corr_df),
        "pearson_corr": corr_df["village_top_share"].corr(corr_df["llm_vote_entropy"]),
    },
])

print("Village vote dispersion vs LLM instability:")
display(village_llm_correlation_df)

print()
print("Per-game human/village/LLM table:")
display(human_llm_game_df.head())


Village vote dispersion vs LLM instability:


,x,y,n_games,pearson_corr
0,village_vote_entropy,llm_vote_entropy,183,-0.008963
1,village_top_share,llm_vote_entropy,183,-0.011688



Per-game human/village/LLM table:


,game_id,actual_human_available,actual_human_vote_type,actual_human_targets,actual_human_target_names,actual_human_win,n_village_aligned_votes,village_vote_counts,village_top_targets,village_top_target_names,...,village_vote_entropy,village_modal_correct,llm_vote_entropy,llm_top_share,agreement_type,stochastic_majority_vote,has_stochastic_majority,stochastic_majority_correct,greedy_vote,greedy_correct
0,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,True,player,"[2, 4]","[Paul, Mitchell]",False,4,"{'Paul': 1, 'Mitchell': 2, 'Justin': 1}",[4],[Mitchell],...,1.039721,False,0.636514,0.666667,two_vs_one,Mitchell,True,False,Justin,True
1,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,True,player,[3],[James],False,4,"{'James': 3, 'Mitchell': 1}",[3],[James],...,0.562335,False,-0.000000,1.000000,unanimous,James,True,False,James,False
2,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,True,player,[0],[Justin],False,4,"{'Laura': 1, 'Paul': 1, 'James': 1, 'Justin': 1}","[1, 2, 3, 0]","[Laura, Paul, James, Justin]",...,1.386294,False,1.098612,0.333333,all_different,NaN,False,NaN,No Werewolf,False
3,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,True,player,[1],[Laura],False,5,"{'Laura': 3, 'Paul': 1, 'Justin': 1}",[1],[Laura],...,0.950271,False,1.098612,0.333333,all_different,NaN,False,NaN,Paul,False
4,Youtube / Flashback##ONE#NIGHT#ULTIMATE#WEREWO...,True,player,[4],[Mitchell],False,3,"{'Mitchell': 2, 'Justin': 1}",[4],[Mitchell],...,0.636514,False,-0.000000,1.000000,unanimous,Mitchell,True,False,Mitchell,False


## Human/village success vs LLM success

In [15]:

def summarize_llm_success_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Summarize stochastic-majority and greedy correctness by a human/village grouping."""
    work = df.dropna(subset=[group_col]).copy()

    if work.empty:
        return pd.DataFrame(columns=[
            group_col,
            "n_games",
            "stochastic_majority_accuracy",
            "greedy_accuracy",
        ])

    return (
        work
        .groupby(group_col, dropna=False)
        .agg(
            n_games=("game_id", "count"),
            stochastic_majority_accuracy=("stochastic_majority_correct", "mean"),
            greedy_accuracy=("greedy_correct", "mean"),
        )
        .reset_index()
    )


human_llm_success_by_actual_human_df = summarize_llm_success_by_group(
    human_llm_game_df,
    "actual_human_win",
)

human_llm_success_by_village_df = summarize_llm_success_by_group(
    human_llm_game_df,
    "village_modal_correct",
)

for df in [human_llm_success_by_actual_human_df, human_llm_success_by_village_df]:
    if not df.empty:
        df["stochastic_majority_accuracy_percent"] = df["stochastic_majority_accuracy"] * 100
        df["greedy_accuracy_percent"] = df["greedy_accuracy"] * 100

print("LLM success grouped by actual human outcome:")
display(human_llm_success_by_actual_human_df)

print()
print("LLM success grouped by village-aligned modal correctness:")
display(human_llm_success_by_village_df)


def human_llm_outcome_bucket(row: pd.Series) -> str:
    """Bucket games by actual human outcome and stochastic-majority outcome."""
    if (
        pd.isna(row["actual_human_win"])
        or not bool(row.get("has_stochastic_majority", False))
        or pd.isna(row["stochastic_majority_correct"])
    ):
        return "missing_or_no_majority"

    human_win = bool(row["actual_human_win"])
    llm_win = bool(row["stochastic_majority_correct"])

    if human_win and llm_win:
        return "human_win_llm_win"
    if human_win and not llm_win:
        return "human_win_llm_lose"
    if (not human_win) and llm_win:
        return "human_lose_llm_win"
    return "human_lose_llm_lose"


human_llm_game_df["human_llm_outcome_bucket"] = human_llm_game_df.apply(
    human_llm_outcome_bucket,
    axis=1,
)

human_llm_outcome_overlap_df = (
    human_llm_game_df["human_llm_outcome_bucket"]
    .value_counts()
    .rename_axis("bucket")
    .reset_index(name="n_games")
)

comparable_human_llm_df = human_llm_game_df[
    human_llm_game_df["human_llm_outcome_bucket"] != "missing_or_no_majority"
].copy()

comparable_human_llm_df["same_human_llm_outcome"] = (
    comparable_human_llm_df["actual_human_win"].astype(bool)
    == comparable_human_llm_df["stochastic_majority_correct"].astype(bool)
)

unanimity_overlap_df = (
    comparable_human_llm_df
    .groupby("agreement_type")
    .agg(
        n_games=("game_id", "count"),
        same_outcome_n=("same_human_llm_outcome", "sum"),
        same_outcome_rate=("same_human_llm_outcome", "mean"),
    )
    .reset_index()
)
unanimity_overlap_df["same_outcome_rate_percent"] = unanimity_overlap_df["same_outcome_rate"] * 100

same_wrong_candidate_df = human_llm_game_df[
    (human_llm_game_df["village_modal_correct"] == False)
    & (human_llm_game_df["stochastic_majority_correct"] == False)
    & (human_llm_game_df["has_stochastic_majority"] == True)
].copy()

same_wrong_candidate_df["same_wrong_candidate"] = same_wrong_candidate_df.apply(
    lambda row: row["stochastic_majority_vote"] in set(row["village_top_target_names"])
    if isinstance(row["village_top_target_names"], list)
    else False,
    axis=1,
)

same_wrong_candidate_summary_df = pd.DataFrame([{
    "n_village_wrong_and_llm_wrong": len(same_wrong_candidate_df),
    "n_same_wrong_candidate": int(same_wrong_candidate_df["same_wrong_candidate"].sum()) if len(same_wrong_candidate_df) else 0,
    "same_wrong_candidate_rate": same_wrong_candidate_df["same_wrong_candidate"].mean() if len(same_wrong_candidate_df) else np.nan,
}])
same_wrong_candidate_summary_df["same_wrong_candidate_rate_percent"] = (
    same_wrong_candidate_summary_df["same_wrong_candidate_rate"] * 100
)

print("Human/LLM outcome overlap:")
display(human_llm_outcome_overlap_df)

print()
print("Relation between LLM agreement type and human/LLM outcome overlap:")
display(unanimity_overlap_df)

print()
print("Same wrong candidate between village modal preference and stochastic majority:")
display(same_wrong_candidate_summary_df)


LLM success grouped by actual human outcome:


,actual_human_win,n_games,stochastic_majority_accuracy,greedy_accuracy,stochastic_majority_accuracy_percent,greedy_accuracy_percent
0,False,87,0.25641,0.321839,25.641026,32.183908
1,True,84,0.6125,0.571429,61.25,57.142857



LLM success grouped by village-aligned modal correctness:


,village_modal_correct,n_games,stochastic_majority_accuracy,greedy_accuracy,stochastic_majority_accuracy_percent,greedy_accuracy_percent
0,False,75,0.246377,0.280000,24.637681,28.000000
1,True,108,0.574257,0.564815,57.425743,56.481481


Human/LLM outcome overlap:


,bucket,n_games
0,human_lose_llm_lose,58
1,human_win_llm_win,49
2,missing_or_no_majority,33
3,human_win_llm_lose,31
4,human_lose_llm_win,20



Relation between LLM agreement type and human/LLM outcome overlap:


,agreement_type,n_games,same_outcome_n,same_outcome_rate,same_outcome_rate_percent
0,two_vs_one,52,30,0.576923,57.692308
1,unanimous,106,77,0.726415,72.641509



Same wrong candidate between village modal preference and stochastic majority:


,n_village_wrong_and_llm_wrong,n_same_wrong_candidate,same_wrong_candidate_rate,same_wrong_candidate_rate_percent
0,52,29,0.557692,55.769231


## Exports


In [17]:
def make_csv_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Convert list/dict/tuple cells to JSON strings before saving CSVs."""
    out = df.copy()

    def convert_value(value: Any) -> Any:
        if isinstance(value, (list, dict, tuple)):
            return json.dumps(value, ensure_ascii=False)
        return value

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].map(convert_value)

    return out


def save_table(df: pd.DataFrame, path: Path) -> None:
    """Save a dataframe as CSV, creating parent directories."""
    path.parent.mkdir(parents=True, exist_ok=True)
    make_csv_safe(df).to_csv(path, index=False)


def save_printable_examples(
    sampled_examples_df: pd.DataFrame,
    examples_dir: Path,
    include_justifications: bool = True,
) -> None:
    """Save sampled examples as readable .txt files, grouped by agreement case."""
    examples_dir.mkdir(parents=True, exist_ok=True)

    if sampled_examples_df.empty:
        (examples_dir / "sampled_examples_all_cases.txt").write_text(
            "No sampled agreement examples available.\n",
            encoding="utf-8",
        )
        return

    all_case_texts = []

    for agreement_case, group in sampled_examples_df.groupby("agreement_case"):
        game_blocks = [
            format_game_example(game_id, include_justifications=include_justifications)
            for game_id in group["game_id"]
        ]

        case_text = "\n\n".join([
            "=" * 88,
            f"AGREEMENT CASE: {agreement_case}",
            f"N EXAMPLES: {len(group)}",
            "=" * 88,
            "",
            "\n\n".join(game_blocks),
            "",
        ])

        safe_case_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", agreement_case).strip("_")
        case_path = examples_dir / f"{safe_case_name}.txt"
        case_path.write_text(case_text, encoding="utf-8")

        all_case_texts.append(case_text)

    (examples_dir / "sampled_examples_all_cases.txt").write_text(
        "\n\n".join(all_case_texts),
        encoding="utf-8",
    )


def export_analysis_outputs() -> dict[str, Path]:
    """Save reusable analysis outputs produced by the notebook."""
    exported = {}

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

    config = {
        "llm_name": LLM_NAME,
        "model_stage": MODEL_STAGE,
        "prompt_family": PROMPT_FAMILY,
        "prompt_family_clean": PROMPT_FAMILY_CLEAN,
        "circle_labels": sorted(CIRCLE_LABELS),
        "repo_root": str(REPO_ROOT),
        "idx_file_path": str(IDX_FILE_PATH),

        # Input results
        "results_dir": str(RESULTS_DIR),
        "model_results_root": str(MODEL_RESULTS_ROOT),
        "runs_root": str(RUNS_ROOT),
        "runs_root_used": str(RUNS_ROOT_USED),

        # Notebook analysis outputs
        "analysis_dir": str(ANALYSIS_DIR),
        "model_analysis_root": str(MODEL_ANALYSIS_ROOT),
        "stage_analysis_root": str(STAGE_ANALYSIS_ROOT),
        "voting_analysis_root": str(VOTING_ANALYSIS_ROOT),
        "out_dir": str(OUT_DIR),

        # Sampling
        "random_state": RANDOM_STATE,
        "sample_per_agreement_type": SAMPLE_PER_AGREEMENT_TYPE,
    }

    config_path = OUT_DIR / "config.json"
    save_json(config, config_path)
    exported["config.json"] = config_path

    tables_to_export = {
        "ground_truth_games.csv": "gt_df",
        "run_directories.csv": "run_dirs_df",
        "llm_vote_file_level.csv": "llm_vote_df",
        "llm_run_summary.csv": "llm_run_summary_df",
        "status_counts_by_run.csv": "status_counts_df",

        "vote_matrix.csv": "vote_matrix",
        "pairwise_agreement_stochastic.csv": "pairwise_agreement_df",
        "agreement_type_summary_stochastic.csv": "agreement_type_summary_df",
        "complete_agreement_stochastic.csv": "complete_agreement_df",
        "disagreements_stochastic.csv": "disagreements_df",
        "agreement_cases_stochastic.csv": "agreement_cases_df",
        "sampled_agreement_examples.csv": "sampled_agreement_examples_df",

        "stochastic_majority.csv": "stochastic_majority_df",
        "stochastic_correctness.csv": "stochastic_correctness_df",
        "aggregation_summary.csv": "aggregation_summary_df",

        "greedy_vs_stochastic.csv": "greedy_vs_stochastic_df",
        "greedy_vs_stochastic_summary.csv": "greedy_vs_stochastic_summary_df",

        "human_summary.csv": "human_summary_df",
        "human_per_game.csv": "human_per_game_df",
        "village_vote_dispersion.csv": "village_vote_df",
        "village_llm_correlation.csv": "village_llm_correlation_df",

        "human_llm_game.csv": "human_llm_game_df",
        "human_llm_success_by_actual_human.csv": "human_llm_success_by_actual_human_df",
        "human_llm_success_by_village.csv": "human_llm_success_by_village_df",
        "human_llm_outcome_overlap.csv": "human_llm_outcome_overlap_df",
        "unanimity_overlap.csv": "unanimity_overlap_df",

        "same_wrong_candidate.csv": "same_wrong_candidate_df",
        "same_wrong_candidate_summary.csv": "same_wrong_candidate_summary_df",

        "comparison_with_human.csv": "comparison_with_human_df",
    }

    if SAVE_TABLES:
        for filename, variable_name in tables_to_export.items():
            table = globals().get(variable_name)

            if table is None:
                print(f"Skipping {filename}: {variable_name} is not defined.")
                continue

            if not isinstance(table, pd.DataFrame):
                print(f"Skipping {filename}: {variable_name} is not a dataframe.")
                continue

            table_to_save = table.reset_index() if filename == "vote_matrix.csv" else table
            path = TABLES_DIR / filename

            save_table(table_to_save, path)
            exported[filename] = path

    if SAVE_PRINTABLE_EXAMPLES:
        sampled = globals().get("sampled_agreement_examples_df")

        if sampled is None:
            print("Skipping printable examples: sampled_agreement_examples_df is not defined.")
        else:
            save_printable_examples(sampled, EXAMPLES_DIR)
            exported["sampled_examples_all_cases.txt"] = (
                EXAMPLES_DIR / "sampled_examples_all_cases.txt"
            )

    return exported


exported_paths = export_analysis_outputs()

print(f"Saved {len(exported_paths)} output artifacts under:")
print(OUT_DIR)
print()

for name, path in exported_paths.items():
    print(f"{name}: {path}")

Saved 31 output artifacts under:
C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability

config.json: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\config.json
ground_truth_games.csv: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\ground_truth_games.csv
run_directories.csv: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\run_directories.csv
llm_vote_file_level.csv: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\base\voting\prompt_v4\vote_stability\tables\llm_vote_file_level.csv
llm_run_summary.csv: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\unsloth_gemma-4-3